# Fine-Tuning Laya with TorchTPU on Colab

[![PyPI version](https://img.shields.io/pypi/v/laya.svg)](https://pypi.org/project/laya/)
[![Hugging Face Model](https://img.shields.io/badge/%F0%9F%A4%97%20Model-convaiinnovations%2Flaya-blue)](https://huggingface.co/convaiinnovations/laya)
[![Dataset](https://img.shields.io/badge/%F0%9F%A4%97%20Dataset-LocalLLaMA%2Ftyped--decisions-green)](https://huggingface.co/datasets/LocalLLaMA/typed-decisions)

This notebook fine-tunes **Laya** (`convaiinnovations/laya`, 421M params) on the **1,200 training cases (6,000 typed decisions)** of the [LocalLLaMA/typed-decisions](https://huggingface.co/datasets/LocalLLaMA/typed-decisions) benchmark using **TPU v5e-1 with TorchTPU 0.2.0**

Notebook Settings:
- Runtime → Change runtime type: `v5e-1 TPU`
- Output: Checkpoints and final model are saved to `/content/laya_finetuned_typed_decisions`

Changed lines are marked `# TorchTPU:`; the training recipe is unchanged.
- Device `"tpu"`, process group `tpu_dist`, `torch.compile(backend="tpu")`.
- fp32 without autocast or `GradScaler` (bf16 autocast fails on `nn.LayerNorm` in 0.2.0,
  [#5678](https://github.com/google-pytorch/torch_tpu/issues/5678)).
- Fixed shapes, in training and evaluation, so the TPU compiles a few programs, not one per batch.
- Pinned package versions. The Hub upload (section 8) is off (`PUSH = False`).


## 1. Environment & TPU v5e-1 Check
Verify the TPU chips are detected.


In [ ]:
# TorchTPU: install torch_tpu and check the TPU.
!pip install -q torch==2.14.1 torch_tpu==0.2.0 numpy==2.5.4 --extra-index-url https://download.pytorch.org/whl/cpu
!pip uninstall -y -q torchvision torchaudio

import os, sys
if "numpy" in sys.modules and sys.modules["numpy"].__version__ != "2.5.4":
    print("numpy 2.5.4 was installed under a running kernel: restarting the runtime. "
          "Run all again (Runtime -> Run all); the install is then a no-op.")
    os.kill(os.getpid(), 9)
    
    
import os, subprocess

# In a subprocess, so that this kernel does not hold the TPU the training workers need.
out = subprocess.run(
    ["python", "-c", "import torch; p = torch.tpu.get_device_properties(); "
     "print('TPUCHECK', torch.accelerator.device_count(), "
     "getattr(p, 'hbm_bytes_per_device', None) or p.total_memory, p.name)"],
    capture_output=True, text=True)
line = next((ln for ln in out.stdout.splitlines() if ln.startswith("TPUCHECK")), None)

assert line, out.stdout[-2000:] + out.stderr[-2000:]
_, n_tpu, hbm, name = line.split(maxsplit=3)
print(f"TPU Available: {int(n_tpu) > 0} | Visible TPU chips: {n_tpu} | {name} ({int(hbm) / 1e9:.1f} GB each)")

assert int(n_tpu) >= 1, (
    f"Expected 1 TPU chips, but detected {n_tpu}!\n"
    "Please switch the Colab runtime: Runtime -> Change runtime type -> v5e-1 TPU."
)
print("TPU v5e-1 verified and ready for training!")

## 2. Install Dependencies


In [ ]:
!pip install -q "numpy==2.5.4" "laya==0.4.17" "transformers==5.21.0" "datasets==5.1.0" \
                "safetensors==0.8.0" "huggingface_hub==2.3.0" "pyarrow==26.0.0" "pandas==3.1.0" \
                "scipy==1.18.1" "accelerate==1.15.0" "tabulate==0.10.0"  # pinned versions

import os
os.environ.setdefault("LAYA_SEED", "42")  
os.environ["TORCH_DEVICE_BACKEND_AUTOLOAD"] = "0"  # TorchTPU: leave the TPU to the workers

import laya, transformers, datasets, torch
print("Laya version        :", laya.__version__)
print("Transformers version:", transformers.__version__)
print("PyTorch version     :", torch.__version__)

## 3. Download & Preprocess Data for DDP
We preprocess all 1,200 training cases into tokenized items and save them to disk so both DDP worker ranks can read them.


In [ ]:
import os, json, torch
from datasets import load_dataset
from transformers import AutoTokenizer
from huggingface_hub import snapshot_download
from laya.agent import _fix_tokenizer_config
from laya.train import items_from_rows

MODEL_ID = "convaiinnovations/laya"
print(f"Fetching tokenizer and config from {MODEL_ID}...")
model_dir = snapshot_download(MODEL_ID)
_fix_tokenizer_config(model_dir)

tok = AutoTokenizer.from_pretrained(os.path.join(model_dir, "tokenizer"))
with open(os.path.join(model_dir, "rl_agent_config.json")) as f:
    cfg = json.load(f)

print("Downloading LocalLLaMA/typed-decisions (train split)...")
ds_train = load_dataset("LocalLLaMA/typed-decisions", "all", split="train")

rows = []
for row in ds_train:
    rows.append({
        "state": json.loads(row["state"]),
        "questions": json.loads(row["questions"]),
        "gold": json.loads(row["gold"]),
    })

items, skipped = items_from_rows(tok, rows, cfg["max_len"], cfg["head_max_len"])
print(f"Preprocessed {len(items)} training items (skipped {skipped}) across 1,200 cases.")
torch.save(items, "/content/train_items.pt")
print("Saved preprocessed items to /content/train_items.pt")


## 4. DDP Training Script (`train_ddp.py`)
We write the multi-GPU distributed RLCD training script using pure policy gradients with proper scoring rules and DDP gradient synchronization.


In [ ]:
%%writefile /content/tpu_patches.py
"""TorchTPU helpers for train_ddp.py and the evaluation cell."""
import torch.nn.functional as F


def expand_sdpa_query_broadcast():
    """Expand [B, 1, 1, S] attention masks over the query dim (torch_tpu#5677)."""
    sdpa = F.scaled_dot_product_attention
    if getattr(sdpa, "_laya_expanded", False):
        return

    def patched(query, key, value, attn_mask=None, *args, **kwargs):
        if attn_mask is not None and attn_mask.dim() >= 2 and attn_mask.shape[-2] == 1 and query.shape[-2] > 1:
            attn_mask = attn_mask.expand(*attn_mask.shape[:-2], query.shape[-2], attn_mask.shape[-1]).contiguous()
        return sdpa(query, key, value, attn_mask, *args, **kwargs)

    patched._laya_expanded = True
    F.scaled_dot_product_attention = patched


import torch


def _pad_to(n, step):
    return -(-n // step) * step


def bucket_agent(agent, len_step=128, k_step=4, rows_step=8):
    """Wrap `agent._infer` so every forward runs on a bucketed shape. Returns the agent."""
    if getattr(agent, "_bucketed", False):
        return agent
    infer = agent._infer
    pad_id = agent.tok.pad_token_id or 0

    def bucketed(b):
        ids = b["input_ids"]
        n, length = ids.shape
        k = b["marker_pos"].shape[1]
        n2, l2, k2 = _pad_to(n, rows_step), _pad_to(length, len_step), _pad_to(k, k_step)
        if (n2, l2, k2) == (n, length, k):
            return infer(b)
        p = dict(b)
        p["input_ids"] = torch.full((n2, l2), pad_id, dtype=ids.dtype)
        p["input_ids"][:n, :length] = ids
        p["attention_mask"] = torch.zeros((n2, l2), dtype=b["attention_mask"].dtype)
        p["attention_mask"][:n, :length] = b["attention_mask"]
        # Padded rows attend to one token so they stay finite; they are dropped below.
        p["attention_mask"][n:, 0] = 1
        p["marker_pos"] = torch.zeros((n2, k2), dtype=b["marker_pos"].dtype)
        p["marker_pos"][:n, :k] = b["marker_pos"]
        p["marker_mask"] = torch.zeros((n2, k2), dtype=b["marker_mask"].dtype)
        p["marker_mask"][:n, :k] = b["marker_mask"]
        p["marker_mask"][n:, 0] = True
        p["qtype"] = torch.zeros((n2,), dtype=b["qtype"].dtype)
        p["qtype"][:n] = b["qtype"]
        for key in ("position_ids", "option_ids"):
            if key in b:
                raise ValueError("bucket_agent supports the sequential option layout only")
        logits, act = infer(p)
        return logits[:n, :k], act[:n]

    agent._infer = bucketed
    agent._bucketed = True
    return agent


In [ ]:
%%writefile /content/train_ddp.py
"""TorchTPU DDP training script for the typed-decisions fine-tune (LocalLLaMA/typed-decisions v5e-1).

Training semantics -- item construction, encoding/collation, the RLCD objective,
the sigma schedule, the calibration holdout and the temperature fit -- come from
the canonical primitives in `laya.train` / `laya.common` / `laya.calibrate`.
What intentionally stays notebook-private is the distributed orchestration:
process-group setup, rank/local-rank wiring, the DDP wrapper, equal-rank
sharding and the effective-batch accounting.
"""
import os, sys, time, json, random, math
import numpy as np
import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from safetensors.torch import load_file
from transformers import AutoTokenizer
import torch_tpu  # noqa: F401  # TorchTPU
sys.path.insert(0, "/content")
from tpu_patches import expand_sdpa_query_broadcast

from laya.common import build_model, collate_items
from laya.calibrate import fit_temperature_map
from laya.train import (
    calibration_records,
    encode_item,
    rlcd_loss,
    save_checkpoint,
    sigma_at,
    split_calibration,
)

def pad_to_bucket(batch, pad_id, len_step, max_len, kmax):
    """TorchTPU: pad a batch to a length bucket and the max option count (padding is masked)."""
    pad = torch.nn.functional.pad
    L, k = batch["input_ids"].shape[1], batch["marker_pos"].shape[1]
    L2 = min(max_len, len_step * math.ceil(L / len_step))
    batch["input_ids"] = pad(batch["input_ids"], (0, L2 - L), value=pad_id)
    batch["attention_mask"] = pad(batch["attention_mask"], (0, L2 - L))
    for key in ("marker_pos", "marker_mask", "target"):
        batch[key] = pad(batch[key], (0, kmax - k))
    return batch

def main():
    dist.init_process_group("tpu_dist")  # TorchTPU: one process per chip
    rank = dist.get_rank()
    world_size = dist.get_world_size()
    device = torch.device("tpu")
    torch.tpu.eager_mode = torch.tpu.EagerMode.DEFER_AND_FUSE
    expand_sdpa_query_broadcast()
    LEN_STEP = 128  # TorchTPU: sequence-length bucket

    model_dir = sys.argv[1]
    output_dir = sys.argv[2]

    with open(os.path.join(model_dir, "rl_agent_config.json")) as f:
        cfg = json.load(f)
    cfg["gradient_checkpointing"] = True
    cfg["max_tokens_per_batch"] = 4096
    cfg["max_len"] = 1024
    cfg["head_max_len"] = 256
    max_len = cfg["max_len"]
    head_max_len = cfg["head_max_len"]

    tok = AutoTokenizer.from_pretrained(os.path.join(model_dir, "tokenizer"))
    model = build_model(cfg, encoder_dir=os.path.join(model_dir, "encoder"))

    weights = load_file(os.path.join(model_dir, "model.safetensors"))
    model.load_state_dict(weights, strict=True)

    model.encoder.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.head_checkpointing = True
    model.to(device)
    model.train()

    ddp_model = DDP(model, find_unused_parameters=True)  # TorchTPU
    ddp_model = torch.compile(ddp_model, backend="tpu", dynamic=False)  # TorchTPU

    all_items = torch.load("/content/train_items.pt", weights_only=False)

    # Held out before sharding: `split_calibration` is seed-deterministic and
    # rank-independent, so every rank withholds the same items, and the shard
    # split happens after -- the calibration slice never reaches a training batch.
    train_items, calib_items = split_calibration(all_items, calib_max=400, calib_frac=0.1, seed=20260922)

    # Trim before sharding so every rank gets the same count (drops at most
    # world_size-1 items). Otherwise ranks can run different numbers of
    # micro-batches and DDP deadlocks on the unmatched all-reduce (#678).
    train_items = train_items[:len(train_items) // world_size * world_size]
    my_items = train_items[rank::world_size]
    kmax_all = max(it["k"] for it in all_items)

    EPOCHS = 4
    SEED = int(os.environ.get("LAYA_SEED", "42"))  # instrumentation
    MICRO_BATCH = 8      # 8 sequences per forward pass per GPU
    GRAD_ACCUM = 64 // (MICRO_BATCH * world_size)  # TorchTPU: effective batch 64
    GROUP_SIZE = 4       # RLCD baseline samples
    LR_ENCODER = 2.5e-5  # Encoder adaptation rate
    LR_HEAD = 1.0e-4     # Head adaptation rate
    SIGMA_START = 0.4    # Exploration noise
    SIGMA_END = 0.1

    enc_params = [p for n, p in ddp_model.named_parameters() if "encoder." in n]
    head_params = [p for n, p in ddp_model.named_parameters() if "encoder." not in n]

    optimizer = torch.optim.AdamW([
        {"params": enc_params, "lr": LR_ENCODER},
        {"params": head_params, "lr": LR_HEAD}
    ], weight_decay=0.01)

    total_updates = (len(my_items) // (MICRO_BATCH * GRAD_ACCUM)) * EPOCHS
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, total_updates), eta_min=1e-6)

    if rank == 0:
        print(f"Starting TPU DDP training on {world_size} chips (grad-accum {GRAD_ACCUM}): {len(train_items)} train items ({len(calib_items)} held out for calibration) | {len(my_items)} per rank | {EPOCHS} epochs")
    if rank == 0 and len(my_items) % MICRO_BATCH:
        print(f"Last micro-batch: {len(my_items) % MICRO_BATCH} items, padded to {MICRO_BATCH} (padding is left out of the loss)")
    t0 = time.time()
    epoch_times = []  # instrumentation

    for epoch in range(EPOCHS):
        t_epoch = time.time()  # instrumentation
        random.seed(SEED + epoch + rank)  # instrumentation
        random.shuffle(my_items)
        epoch_loss, n_batches = 0.0, 0
        optimizer.zero_grad(set_to_none=True)
        accum_step = 0

        sigma = sigma_at(epoch, EPOCHS, SIGMA_START, SIGMA_END)
        steps_per_epoch = math.ceil(len(my_items) / MICRO_BATCH)

        for b_idx in range(0, len(my_items), MICRO_BATCH):
            chunk = my_items[b_idx:b_idx + MICRO_BATCH]
            if not chunk:
                continue
            n_real = len(chunk)
            chunk = chunk + chunk[:1] * (MICRO_BATCH - n_real)  # TorchTPU: pad a partial micro-batch

            encoded = [encode_item(tok, it, max_len, head_max_len) for it in chunk]
            batch = pad_to_bucket(collate_items([encoded], tok.pad_token_id), tok.pad_token_id,
                                  LEN_STEP, max_len, kmax_all)  # TorchTPU: fixed shapes

            # TorchTPU: fp32, no autocast (torch_tpu#5678)
            logits, act = ddp_model(
                batch["input_ids"].to(device),
                batch["attention_mask"].to(device),
                batch["marker_pos"].to(device),
                batch["marker_mask"].to(device),
                batch["qtype"].to(device)
            )

            logits = logits.float()
            mask = batch["marker_mask"].to(device)
            target = batch["target"].to(device)
            if n_real < MICRO_BATCH:  # TorchTPU: loss on the real rows only
                logits, mask, target = logits[:n_real], mask[:n_real], target[:n_real]
                batch["qtype"] = batch["qtype"][:n_real]

            # Canonical objective: RLCD policy gradient over noisy logits + full
            # soft cross-entropy guidance (`laya.train.rlcd_loss`).
            loss = rlcd_loss(logits, target, mask, batch["qtype"].to(device), sigma,
                             samples=GROUP_SIZE, w_sph=0.75, w_rps=1.0)

            window_start = (accum_step // GRAD_ACCUM) * GRAD_ACCUM
            window_size = min(GRAD_ACCUM, steps_per_epoch - window_start)
            scaled = loss / window_size
            scaled.backward()  # TorchTPU: fp32, no GradScaler
            accum_step += 1

            if accum_step % GRAD_ACCUM == 0 or (b_idx + MICRO_BATCH) >= len(my_items):
                torch.nn.utils.clip_grad_norm_(ddp_model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)

            epoch_loss += loss.item()
            n_batches += 1

            if rank == 0 and (n_batches % 50) == 0:
                cur_lr = scheduler.get_last_lr()[0]
                print(f"  Epoch {epoch+1}/{EPOCHS} | Step {n_batches} | Loss: {loss.item():.4f} | LR: {cur_lr:.2e}")

        epoch_times.append(time.time() - t_epoch)  # instrumentation
        if rank == 0:
            print(f"=== Epoch {epoch+1}/{EPOCHS} Completed in {time.time()-t0:.1f}s | Avg Loss: {epoch_loss/max(1, n_batches):.4f} ===")

        dist.barrier()

        # Overwrite a single rolling checkpoint after each epoch so a crash,
        # OOM, or Colab session timeout doesn't lose all prior training.
        if rank == 0:
            ckpt_dir = os.path.join(output_dir, "checkpoint_latest")
            save_checkpoint(model, tok, dict(cfg, max_len=max_len, head_max_len=head_max_len), ckpt_dir)
            with open(os.path.join(ckpt_dir, "checkpoint_meta.json"), "w") as f:
                json.dump({
                    "epoch": epoch + 1,
                    "total_epochs": EPOCHS,
                    "avg_loss": epoch_loss / max(1, n_batches)
                }, f, indent=2)
            print(f"  Saved rolling checkpoint (epoch {epoch+1}/{EPOCHS}) to {ckpt_dir}")

    dist.barrier()

    # Instrumentation: epoch times and peak device memory
    try:
        peak_mem = torch.accelerator.max_memory_allocated()
    except Exception:
        peak_mem = None
    if rank == 0:
        os.makedirs(output_dir, exist_ok=True)
        with open(os.path.join(output_dir, "train_timing.json"), "w") as f:
            json.dump({
                "world_size": world_size,
                "seed": SEED,
                "device": str(device),
                "epochs_s": [round(t, 1) for t in epoch_times],
                "first_epoch_s": round(epoch_times[0], 1),
                "mean_epoch_s": round(sum(epoch_times) / len(epoch_times), 1),
                "mean_epoch_after_first_s": round(sum(epoch_times[1:]) / max(1, len(epoch_times) - 1), 1),
                "total_epochs_s": round(sum(epoch_times), 1),
                "total_train_s": round(time.time() - t0, 1),
                "peak_device_memory_gb_rank0": None if peak_mem is None else round(peak_mem / 1e9, 2),
            }, f, indent=2)

    # Post-training temperature calibration on rank 0
    if rank == 0:
        print("\nFitting post-training calibration temperatures...")
        del optimizer, scheduler  # TorchTPU
        model.eval()
        try:
            records = calibration_records(model, tok, calib_items, device, max_len, head_max_len, batch_size=16)
            fitted = fit_temperature_map(records)
            fitted_temps = fitted["temperature"]
            print("Fitted calibration temperatures (choice, score, noul):", [round(t, 3) for t in fitted_temps])
        except Exception as e:
            print("Temperature fitting fallback:", e)
            fitted_temps, fitted = None, {"temperature_by_options": {}}

        out_cfg = dict(cfg, max_len=max_len, head_max_len=head_max_len)
        out_cfg["fine_tuned"] = True
        out_cfg["model_name"] = "laya-typed-decisions"
        if fitted_temps is not None:
            out_cfg["temperature"] = fitted_temps
            # This fit carries per-bucket overrides; inherited ones would shadow them.
            out_cfg.pop("temperature_by_options", None)
            if fitted["temperature_by_options"]:
                out_cfg["temperature_by_options"] = fitted["temperature_by_options"]
        save_checkpoint(model, tok, out_cfg, output_dir)
        print(f"Model successfully saved to {output_dir}!")
        open(os.path.join(output_dir, ".rank0_done"), "w").close()
    else:
        # TorchTPU: wait for rank 0; a worker that exits early aborts the TPU slice.
        while not os.path.exists(os.path.join(output_dir, ".rank0_done")):
            time.sleep(5)

    dist.destroy_process_group()

if __name__ == "__main__":
    main()


## 5. Launch Multi-GPU Fine-Tuning with `torchrun`
Runs one process per TPU chip (1). The first epoch includes TPU compilation.


In [ ]:
OUTPUT_DIR = "/content/laya_finetuned_typed_decisions"
MODEL_DIR = model_dir

# TorchTPU: torchrun settings from torch.tpu.distributed
cmd = ("export TORCH_DEVICE_BACKEND_AUTOLOAD=1 && "
       "eval $(python -m torch.tpu.distributed | sed 's/^/export /') && "
       f"torchrun --nproc_per_node=$WORLD_SIZE /content/train_ddp.py {MODEL_DIR} {OUTPUT_DIR}")
print("Executing DDP training:", cmd)
# TorchTPU: run in bash and stream the output
import subprocess
proc = subprocess.Popen(cmd, shell=True, executable="/bin/bash", stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="", flush=True)
assert proc.wait() == 0, f"training failed with exit code {proc.returncode}"


## 6. Run Benchmark Evaluation on Test Split (400 cases / 2,000 decisions)
Evaluate the fine-tuned model against the official `test` split.


In [ ]:
import time, json
import numpy as np
import pandas as pd
from datasets import load_dataset
import laya

# TorchTPU: load the backend (autoload is off in this kernel)
import os, torch_tpu, torch_tpu._loader
os.environ["TORCH_DEVICE_BACKEND_AUTOLOAD"] = "1"
torch_tpu._loader.load()
import sys; sys.path.insert(0, "/content")
from tpu_patches import bucket_agent, expand_sdpa_query_broadcast
expand_sdpa_query_broadcast()
from laya.common import ece_score

print("Loading test split for evaluation...")
ds_test = load_dataset("LocalLLaMA/typed-decisions", "all", split="test")

agent_ft = laya.Agent(OUTPUT_DIR, device="tpu")  # TorchTPU
bucket_agent(agent_ft)  # TorchTPU: fixed shapes

predictions = []
latencies_ms = []

print(f"Evaluating {len(ds_test)} test cases on TPU (the first cases compile)...")
t0_eval = time.time()

for i, row in enumerate(ds_test):
    case_id = row["id"]
    workflow = row["workflow"]
    state = json.loads(row["state"])
    questions = json.loads(row["questions"])
    gold = json.loads(row["gold"])
    
    t0 = time.perf_counter()
    res = agent_ft.predict(state, questions)
    dt_ms = (time.perf_counter() - t0) * 1000
    latencies_ms.append(dt_ms)
    
    predictions.append({
        "id": case_id,
        "workflow": workflow,
        "pred": res["answers"],
        "gold": gold,
        "questions": questions,
        "latency_ms": dt_ms
    })

print(f"Evaluated all {len(ds_test)} cases in {time.time() - t0_eval:.1f}s!")
eval_s = time.time() - t0_eval  # instrumentation


## 7. Compute Official Metrics & Comparison Table against TypeSafe Jev


In [ ]:
accuracies = []
soft_accuracies = []
brier_scores = []
kl_divs = []
tv_distances = []
score_maes = []
within_one = []
all_confs = []
all_corrects = []

for item in predictions:
    pred_answers = item["pred"]
    gold_answers = item["gold"]
    questions = item["questions"]
    
    for qid, qdef in questions.items():
        p_ans = pred_answers[qid]
        g_ans = gold_answers[qid]
        q_type = qdef["type"]
        
        # 1. Choice
        if q_type == "choice":
            keys = list(qdef["criteria"].keys())
            pred_choice = p_ans["choice"]
            gold_label = str(g_ans["label"])
            
            is_corr = float(pred_choice == gold_label)
            accuracies.append(is_corr)
            all_corrects.append(is_corr)
            
            p_probs = np.array([p_ans["probabilities"].get(k, 1e-6) for k in keys])
            g_probs = np.array([g_ans["probabilities"].get(k, 1e-6) for k in keys])
            p_probs /= p_probs.sum()
            g_probs /= g_probs.sum()
            
            all_confs.append(float(p_probs.max()))
            soft_accuracies.append(float((p_probs * g_probs).sum()))
            brier_scores.append(float(((p_probs - g_probs) ** 2).sum()))
            tv_distances.append(float(0.5 * np.abs(p_probs - g_probs).sum()))
            kl_divs.append(float((g_probs * np.log(np.clip(g_probs / p_probs, 1e-12, 1e4))).sum()))
            
        # 2. Noul
        elif q_type == "noul":
            p_val = p_ans["noul"]
            g_val = g_ans.get("noul", g_ans.get("probabilities", {}).get("true", 0.5))
            gold_label = str(g_ans["label"]).lower()
            
            pred_label = "true" if p_val >= 0.5 else "false"
            is_corr = float(pred_label == gold_label)
            accuracies.append(is_corr)
            all_corrects.append(is_corr)
            all_confs.append(float(max(p_val, 1.0 - p_val)))
            
            p_dist = np.array([1.0 - p_val, p_val])
            g_dist = np.array([1.0 - g_val, g_val])
            
            soft_accuracies.append(float((p_dist * g_dist).sum()))
            brier_scores.append(float(((p_dist - g_dist) ** 2).sum()))
            tv_distances.append(float(0.5 * np.abs(p_dist - g_dist).sum()))
            kl_divs.append(float((g_dist * np.log(np.clip(g_dist / p_dist, 1e-12, 1e4))).sum()))
            
        # 3. Score
        elif q_type == "score":
            p_score = p_ans["score"]
            g_score = g_ans.get("score", 0.0)
            score_maes.append(abs(p_score - g_score))
            within_one.append(float(abs(p_score - g_score) <= 1.0))
            
            # Use argmax of probabilities for discrete prediction
            n_levels = len(qdef.get("criteria", []))
            p_probs_score = np.array([p_ans["probabilities"].get(str(i), 0.0) for i in range(n_levels)])
            if p_probs_score.sum() > 0:
                p_probs_score /= p_probs_score.sum()
                p_lvl = int(np.argmax(p_probs_score))
                all_confs.append(float(p_probs_score.max()))
            else:
                p_lvl = int(round(p_score))
                all_confs.append(0.5)
                
            g_lvl = int(g_ans.get("label", int(round(g_score))))
            is_corr = float(p_lvl == g_lvl)
            accuracies.append(is_corr)
            all_corrects.append(is_corr)

laya_acc = np.mean(accuracies)
laya_soft_acc = np.mean(soft_accuracies)
laya_brier = np.mean(brier_scores)
laya_kl = np.mean(kl_divs)
laya_tv = np.mean(tv_distances)
laya_ece = ece_score(np.array(all_confs), np.array(all_corrects))
laya_mae = np.mean(score_maes) if score_maes else 0.0
laya_within1 = np.mean(within_one) if within_one else 0.0
laya_latency = np.percentile(latencies_ms, 50)

comparison_data = [
    {
        "Model": "TypeSafe Jev 1.13.0",
        "Kind": "general",
        "Accuracy": 0.727,
        "Soft Acc": 0.580,
        "Brier": 0.148,
        "ECE": 0.144,
        "Score MAE": 0.391,
        "Within 1 Level": "0.952",
        "ms/case": 710,
        "Cost/Case": "$0.0004 (API)"
    },
    {
        "Model": "Laya (Fine-Tuned TPU v5e-1)",
        "Kind": "fine-tuned",
        "Accuracy": round(laya_acc, 3),
        "Soft Acc": round(laya_soft_acc, 3),
        "Brier": round(laya_brier, 3),
        "ECE": round(laya_ece, 3),
        "Score MAE": round(laya_mae, 3),
        "Within 1 Level": f"{laya_within1:.3f}",
        "ms/case": round(laya_latency, 1),
        "Cost/Case": "$0.00 (Self-Hosted)"
    },
    {
        "Model": "ModernBERT-base (149M)",
        "Kind": "specialist",
        "Accuracy": 0.646,
        "Soft Acc": 0.542,
        "Brier": 0.119,
        "ECE": 0.179,
        "Score MAE": 0.444,
        "Within 1 Level": "0.931",
        "ms/case": 349,
        "Cost/Case": "$0.00"
    },
    {
        "Model": "Teacher Self-Agreement",
        "Kind": "ceiling",
        "Accuracy": 0.735,
        "Soft Acc": "-",
        "Brier": "-",
        "ECE": "-",
        "Score MAE": "-",
        "Within 1 Level": "-",
        "ms/case": "-",
        "Cost/Case": "-"
    }
]

df_comp = pd.DataFrame(comparison_data)
print("=== HEAD-TO-HEAD BENCHMARK TABLE ===\n")
print(df_comp.to_markdown(index=False))


## 8. (Optional) Push to Hugging Face Hub


In [ ]:
import os
from huggingface_hub import HfApi

try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
except Exception:
    token = None

PUSH = False  # set True to upload (needs HF_TOKEN)

if PUSH and not token:
    raise RuntimeError(
        "No HF_TOKEN found. Add it in Colab's Secrets panel (key icon, left sidebar) as HF_TOKEN "
        "before running this cell, or set token = \"<your-write-token>\" manually above."
    )

NEW_REPO = "<your-hf-username>/laya-typed-decisions-torchtpu"  # your repo

# Reference baselines pulled from the comparison table above, so the model
# card's narrative always matches the numbers actually computed this run.
JEV_ACC = next(r["Accuracy"] for r in comparison_data if r["Model"] == "TypeSafe Jev 1.13.0")
CEILING_ACC = next(r["Accuracy"] for r in comparison_data if r["Model"] == "Teacher Self-Agreement")

def describe_vs_baseline(acc, jev_acc=JEV_ACC, ceiling_acc=CEILING_ACC):
    """Builds an accuracy comparison sentence that reflects this run's actual result,
    instead of assuming the fine-tuned model always wins."""
    if acc > ceiling_acc:
        return (f"outperforming **TypeSafe Jev 1.13.0 ({jev_acc:.3f})** and surpassing the "
                f"benchmark's **Teacher Self-Agreement ceiling ({ceiling_acc:.3f})**")
    elif acc > jev_acc:
        return (f"outperforming **TypeSafe Jev 1.13.0 ({jev_acc:.3f})**, though still below the "
                f"benchmark's Teacher Self-Agreement ceiling ({ceiling_acc:.3f})")
    else:
        return (f"trailing **TypeSafe Jev 1.13.0 ({jev_acc:.3f})** and the benchmark's "
                f"Teacher Self-Agreement ceiling ({ceiling_acc:.3f})")

comparison_sentence = describe_vs_baseline(laya_acc)

# 1. Write comprehensive model card README.md into OUTPUT_DIR before upload
readme_content = f"""---
license: apache-2.0
library_name: transformers
tags:
- laya
- system-one
- calibrated-decisions
- rlcd
- structured-decisions
- typed-decisions
- benchmark
metrics:
- accuracy
- brier_score
model-index:
- name: laya-typed-decisions
  results:
  - task:
      type: text-classification
      name: System One Decision Benchmark
    dataset:
      type: LocalLLaMA/typed-decisions
      name: Typed Decisions
    metrics:
    - type: accuracy
      value: {laya_acc:.3f}
    - type: brier_score
      value: {laya_brier:.3f}
---

# Laya fine-tuned on Typed-Decisions: TorchTPU port (TPU v5e-1)

> Checkpoint from an unofficial TorchTPU port of the official 2×T4 fine-tuning notebook; not an official Laya release.

This is **Laya** fine-tuned on the 1,200 training cases (6,000 decisions) of the independent [LocalLLaMA/typed-decisions](https://huggingface.co/datasets/LocalLLaMA/typed-decisions) benchmark.

On the official 400-case test set (2,000 decisions across Agent Trace Observability, Customer Service, Invoice Processing, and Security Incidents), it achieves **{laya_acc:.3f} Accuracy**, {comparison_sentence}.

## Head-to-Head Benchmark Results

| Model | Kind | Accuracy | Soft Acc | Brier Score | ECE | Score MAE | Within 1 Level | Latency (p50) | Cost/Case |
|---|---|---|---|---|---|---|---|---|---|
| **Laya (TorchTPU, TPU v5e-1)** | **fine-tuned** | **{laya_acc:.3f}** | **{laya_soft_acc:.3f}** | **{laya_brier:.3f}** | **{laya_ece:.3f}** | **{laya_mae:.3f}** | **{laya_within1:.3f}** | **{laya_latency:.1f} ms** | **$0.00 (Self-Hosted)** |
| TypeSafe Jev 1.13.0 | general | 0.727 | 0.580 | 0.148 | 0.144 | 0.391 | 0.952 | 710 ms | $0.0004 (API) |
| ModernBERT-base (149M) | specialist | 0.646 | 0.542 | 0.119 | 0.179 | 0.444 | 0.931 | 349 ms | $0.00 |
| Teacher Self-Agreement | ceiling | 0.735 | - | - | - | - | - | - | - |

## Installation & Quickstart

```bash
pip install laya
```

```python
import laya

# Load the fine-tuned model directly from Hugging Face
agent = laya.load("{NEW_REPO}")

# Evaluate any workflow state and typed questions in a single forward pass
result = agent.predict(state, questions)
print(result["answers"])
```

## License
Apache 2.0. Laya is developed by [Convai Innovations](https://huggingface.co/convaiinnovations); this checkpoint comes from an unofficial TorchTPU port of their training notebook.
"""

with open(os.path.join(OUTPUT_DIR, "README.md"), "w") as f:
    f.write(readme_content)

print(f"Generated model card at {os.path.join(OUTPUT_DIR, 'README.md')}")

if PUSH:
    print(f"Uploading the {laya_acc:.3f} accuracy checkpoint to {NEW_REPO} (private)...")
    api = HfApi(token=token)
    api.create_repo(NEW_REPO, repo_type="model", private=True, exist_ok=True)

    # Upload the fine-tuned weights, config, tokenizer, and README
    api.upload_folder(
        folder_path=OUTPUT_DIR,
        repo_id=NEW_REPO,
        repo_type="model",
        commit_message=f"Laya fine-tuned on typed-decisions: Accuracy {laya_acc:.3f} vs Jev {JEV_ACC:.3f} / Ceiling {CEILING_ACC:.3f}"
    )

    print(f"\nUploaded (private): https://huggingface.co/{NEW_REPO}")
else:
    print("Skipped the Hub upload (PUSH = False).")


## 9. Save Benchmark Report as JSON
Export the full evaluation metrics, comparison table, and per-workflow accuracy breakdown to JSON.


In [ ]:
import json, os

report = {
    "benchmark": "LocalLLaMA/typed-decisions",
    "model": "Laya (Fine-Tuned TPU v5e-1)",
    "n_cases": len(ds_test),
    "n_decisions": len(ds_test) * 5,
    "metrics": {
        "accuracy": round(float(laya_acc), 4),
        "soft_accuracy": round(float(laya_soft_acc), 4),
        "brier_score": round(float(laya_brier), 4),
        "ece": round(float(laya_ece), 4),
        "score_mae": round(float(laya_mae), 4),
        "within_1_level": round(float(laya_within1), 4),
        "latency_p50_ms": round(float(laya_latency), 1),
        "latency_p95_ms": round(float(np.percentile(latencies_ms, 95)), 1),
        "kl_divergence": round(float(laya_kl), 4),
        "total_variation": round(float(laya_tv), 4)
    },
    "comparison": comparison_data,
    "per_workflow": {}
}

# Instrumentation: training time, versions and evaluation time
_timing = os.path.join(OUTPUT_DIR, "train_timing.json")
if os.path.exists(_timing):
    with open(_timing) as f:
        report["training"] = json.load(f)
import importlib.metadata as _md
def _version(p):
    try:
        return _md.version(p)
    except _md.PackageNotFoundError:
        return None
report["run"] = {"versions": {p: _version(p) for p in ("torch", "torch_tpu", "laya", "transformers", "numpy")},
                 "eval_s": round(eval_s, 1) if "eval_s" in globals() else None}

# Calculate per-workflow accuracy
for wf in ["agent_trace_observability", "customer_service", "invoice_processing", "security_incidents"]:
    wf_items = [item for item in predictions if item["workflow"] == wf]
    wf_corr = []
    for item in wf_items:
        for qid, qdef in item["questions"].items():
            p_ans = item["pred"][qid]
            g_ans = item["gold"][qid]
            if qdef["type"] == "choice":
                wf_corr.append(float(p_ans["choice"] == str(g_ans["label"])))
            elif qdef["type"] == "noul":
                pred_label = "true" if p_ans["noul"] >= 0.5 else "false"
                wf_corr.append(float(pred_label == str(g_ans["label"]).lower()))
            elif qdef["type"] == "score":
                p_probs = [p_ans["probabilities"].get(str(i), 0.0) for i in range(len(qdef.get("criteria", [])))]
                p_lvl = int(np.argmax(p_probs)) if sum(p_probs) > 0 else int(round(p_ans["score"]))
                g_lvl = int(g_ans.get("label", int(round(g_ans.get("score", 0.0)))))
                wf_corr.append(float(p_lvl == g_lvl))
    if wf_corr:
        report["per_workflow"][wf] = {
            "n_decisions": len(wf_corr),
            "accuracy": round(float(np.mean(wf_corr)), 4)
        }

# Save in working directory and model directory
out_file1 = "/content/laya_typed_decisions_benchmark_report.json"
out_file2 = os.path.join(OUTPUT_DIR, "benchmark_report.json")

with open(out_file1, "w") as f:
    json.dump(report, f, indent=2)

with open(out_file2, "w") as f:
    json.dump(report, f, indent=2)

print(f"Benchmark results successfully saved to:")
print(f"  - {out_file1}")
print(f"  - {out_file2}")
print("\nSummary:")
print(json.dumps(report["metrics"], indent=2))
print("\nPer-Workflow Accuracy:")
print(json.dumps(report["per_workflow"], indent=2))
